# get-children-callable-param — ex2: parameters(recurse=True) built on top of get_children with dotted names

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `get-children-callable-param`. Running the final beacon cell reports progress against the `Backprop: get_children callable param` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: get_children callable param` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`get-children-callable-param`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "get-children-callable-param"
DD_SUBTOPIC = "Backprop: get_children callable param"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## get_children → recursive parameters walker — quick refresher

`get_children` is the SHALLOW step (one module's direct attrs). The recursive walk `parameters()` is built on top of it:

```python
class MLP(Module):
    def __init__(self):
        self.fc1 = Linear()       # child Module
        self.fc2 = Linear()       # child Module
        self.bias = MiniTensor(...)

list(MLP().parameters())
  → [('fc1.weight', ...), ('fc1.bias', ...),
     ('fc2.weight', ...), ('fc2.bias', ...),
     ('bias', ...)]
```

Walk: for each (name, val) in `get_children`:
- if val is a `MiniTensor`, yield `(name, val)`
- if val is a `Module`, recurse and prefix each child name with `<name>.`

### Exercise 2 — parameters(recurse=True) built on top of get_children with dotted names

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply the recursive parameter walker pattern: build parameters() on top of get_children by yielding leaf Tensors directly and recursing into Module-valued children with dotted name prefixes.
> Keywords: parameters, recursive, dotted-name, nn.Module, walker
> ```

**KCs targeted:** `get-children-callable-param`, `parameter-subclass-of-tensor`

We've given you a `Module` base class with `get_children` (the ex1 helper, extended to yield BOTH MiniTensors and other `Module` instances). Implement `Module.parameters(self)` as a generator that does a DEPTH-FIRST recursive walk:

- For each `(name, val)` yielded by `get_children`:
  - If `isinstance(val, MiniTensor)`: yield `(name, val)`.
  - If `isinstance(val, Module)`: recurse into `val.parameters()`,     and for each `(sub_name, sub_val)` it yields, yield     `(f'{name}.{sub_name}', sub_val)`.

This produces the canonical PyTorch state-dict naming: `'fc1.weight'`, `'fc1.bias'`, `'fc2.weight'`, etc.

Rules:

**1. Generator (yield).** Same reason as ex1.

**2. Dotted naming.** Prefix child names with the parent attr name + `.`. This is the state-dict convention and is how `load_state_dict` finds the right tensor by key.

**3. Depth-first, in-order.** Order matches `get_children`'s insertion order at each level.

Note: you should NOT call `get_children` from inside `parameters` to discover child Modules separately — the given `get_children` already yields both Tensors and Modules. Just dispatch on type.

In [ ]:
class Module:
    """Tiny nn.Module stand-in. get_children yields BOTH MiniTensors and Modules."""
    def get_children(self):
        for name, val in self.__dict__.items():
            if isinstance(val, (MiniTensor, Module)):
                yield name, val

    def parameters(self):
        """Recursive DFS yield of (dotted_name, MiniTensor) leaves."""
        raise NotImplementedError()


def _test_ex2():
    # --- flat module: parameters == get_children's MiniTensor subset ---
    class Linear(Module):
        def __init__(self):
            self.weight = MiniTensor(t.randn(4, 3), requires_grad=True)
            self.bias = MiniTensor(t.zeros(4), requires_grad=True)
            self.in_features = 3

    lin = Linear()
    params = list(lin.parameters())
    names = [n for n, _ in params]
    assert names == ['weight', 'bias'], f'flat case: {names}'
    assert params[0][1] is lin.weight
    assert params[1][1] is lin.bias

    # --- nested module: parameters yields dotted names ---
    class MLP(Module):
        def __init__(self):
            self.fc1 = Linear()
            self.fc2 = Linear()

    mlp = MLP()
    params = list(mlp.parameters())
    names = [n for n, _ in params]
    assert names == ['fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias'], (
        f'nested case: {names}'
    )
    # identity through nesting
    assert params[0][1] is mlp.fc1.weight
    assert params[2][1] is mlp.fc2.weight

    # --- mixed: top-level MiniTensor alongside child Modules ---
    class MLPWithBias(Module):
        def __init__(self):
            self.fc1 = Linear()
            self.extra_bias = MiniTensor(t.zeros(4), requires_grad=True)
            self.fc2 = Linear()

    m = MLPWithBias()
    names = [n for n, _ in m.parameters()]
    assert names == [
        'fc1.weight', 'fc1.bias', 'extra_bias',
        'fc2.weight', 'fc2.bias',
    ], f'mixed case: {names}'

    # --- three levels of nesting → two dots ---
    class Block(Module):
        def __init__(self):
            self.inner = Linear()

    class Net(Module):
        def __init__(self):
            self.block = Block()

    net = Net()
    names = [n for n, _ in net.parameters()]
    assert names == ['block.inner.weight', 'block.inner.bias'], (
        f'two-level nesting: {names}'
    )

    # --- it is a generator (zero-allocation iteration) ---
    import inspect
    iterator = lin.parameters()
    assert iter(iterator) is iterator or inspect.isgenerator(iterator), (
        f'parameters() must be a generator, got {type(iterator).__name__}'
    )

    # --- empty module: yields nothing (no crash) ---
    class Empty(Module):
        def __init__(self):
            pass
    assert list(Empty().parameters()) == []
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
class Module:
    def get_children(self):
        for name, val in self.__dict__.items():
            if isinstance(val, (MiniTensor, Module)):
                yield name, val

    def parameters(self):
        for name, val in self.get_children():
            if isinstance(val, MiniTensor):
                yield name, val
            elif isinstance(val, Module):
                for sub_name, sub_val in val.parameters():
                    yield f'{name}.{sub_name}', sub_val
```

**Why dotted names.** PyTorch's `state_dict` is a flat dict keyed by these dotted paths. `load_state_dict` matches keys by string. The same convention falls out of recursive prefixing — no extra machinery needed.

**Why depth-first.** Order matters for reproducibility (random init seeds, optimizer state ordering). DFS in attribute-insertion order is what `nn.Module` does in PyTorch.

**Why `get_children` returns BOTH types in ex2.** Ex1 stayed shallow on MiniTensor only. To compose a recursive walker on top, we need both — so `get_children` is extended here. In real `nn.Module` this is split: `_parameters` + `_modules` separately, but the abstraction is the same.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()